# Validate a synthetic instrument log

The input below represents three local records. Run each cell in order.
Parse the CSV, report rejected rows and compute a deterministic summary from
the valid measurements. Then change the fixture and rerun the checks.


In [ ]:
import csv
from decimal import Decimal, InvalidOperation
from io import StringIO

fixture = '''instrument,voltage
sensor-a,1.25
sensor-a,invalid
sensor-b,1.75
'''
rows = list(csv.DictReader(StringIO(fixture)))
assert len(rows) == 3
assert rows[0] == {"instrument": "sensor-a", "voltage": "1.25"}


## Validate measurements

Reason about the distinction between a malformed number and a valid decimal
that represents infinity or NaN. Preserve the row number so an operator can
identify each rejected record. Treat the header as row 1.


In [ ]:
def validate(rows):
    accepted, rejected = [], []
    for row_number, row in enumerate(rows, start=2):
        label = row.get("instrument") or ""
        try:
            value = Decimal(row.get("voltage") or "")
        except InvalidOperation:
            rejected.append(row_number)
            continue
        if not label.strip() or not value.is_finite():
            rejected.append(row_number)
            continue
        accepted.append((label, value))
    return accepted, rejected

accepted, rejected = validate(rows)
assert accepted == [("sensor-a", Decimal("1.25")), ("sensor-b", Decimal("1.75"))]
assert rejected == [3]


## Produce and check a summary

Report a count and arithmetic mean from the valid rows. If none survive
validation, return an empty mean. Use a second test to establish that special
decimal values are rejected before arithmetic.


In [ ]:
def summarize(measurements):
    values = [value for _, value in measurements]
    return {"count": len(values), "mean": sum(values) / len(values) if values else None}

assert summarize(accepted) == {"count": 2, "mean": Decimal("1.50")}
assert summarize([]) == {"count": 0, "mean": None}
non_finite = [{"instrument": "test", "voltage": "NaN"},
              {"instrument": "test", "voltage": "Infinity"}]
assert validate(non_finite) == ([], [2, 3])
assert validate([{"instrument": None, "voltage": None}]) == ([], [2])
print(summarize(accepted))
